# 실습 3-2. 유압 펌프 예지보전

사출기 유압 펌프 4대의 **진동**과 **전류**를 30일 동안 2시간마다 모은 데이터임. 그중 3대가 베어링 고장으로 멈췄음.
고장 전에 나타나는 **전조**를 찾아 경고 · 정지 2단 알람 기준을 만듦.

진행 순서는 상태 감시 표준(ISO 13374)의 여섯 단계를 따름:

| 단계 | 이 노트북에서 |
|---|---|
| 1. 데이터 수집 | 진동 · 전류 불러오기 (1절) |
| 2. 데이터 처리 | 특징 추출: 실효값 · 첨도 · 파고율 (2절) |
| 3. 상태 검출 | 정상 기준과 이상 점수 (3절) |
| 4. 건전성 평가 | 전조를 얼마나 잘 가려내는가 (4절) |
| 5. 예지 평가 | 고장 몇 시간 전에 알 수 있는가 (5절) |
| 6. 조치 권고 | 경고 · 정지 알람 기준표 (6절) |

- **기본 트랙**: 셀을 차례로 실행하고 알람 기준표를 워크북에 옮겨 적음
- **심화 트랙**: `# 심화` 표시가 있는 값을 바꿔 전조 재현율 0.80 이상을 지키면서 오경보를 줄임

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "common").exists():      # 저장소 최상위 폴더 찾기
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from common.plot_style import use_korean_font
use_korean_font()

DATA = ROOT / "data" / "session3"
OUT = DATA / "work"                        # 내 산출물이 저장되는 곳
OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 160)

## 1. 데이터 수집

In [ ]:
from common import process_model as pm

win = pd.read_csv(DATA / "pump_windows.csv", parse_dates=["timestamp"])
vib = np.load(DATA / "pump_vibration.npz")["vibration"].astype(np.float32)      # [구간 수, 1024] 가속도(g), 2 kHz
fail = pd.read_csv(DATA / "failure_log.csv", parse_dates=["failure_time"])
FS = 2000
print("구간:", vib.shape, "  설비:", list(win.unit_id.unique()))
fail

In [ ]:
# 같은 설비의 초기와 고장 직전 파형
u = win[win.unit_id == "PUMP-03"]
t_fail = fail.set_index("unit_id").loc["PUMP-03", "failure_time"]
early, late = u.index[5], u[u.timestamp < t_fail].index[-3]
t = np.arange(vib.shape[1]) / FS * 1000
fig, axes = plt.subplots(1, 2, figsize=(13, 3.2), sharey=True)
for ax, i, name in ((axes[0], early, "초기 (정상)"), (axes[1], late, "고장 6시간 전")):
    ax.plot(t[:600], vib[i][:600], lw=0.8, color="#1f618d"); ax.set_title(f"PUMP-03  {name}"); ax.set_xlabel("시간 (ms)"); ax.grid(alpha=0.3)
axes[0].set_ylabel("가속도 (g)"); plt.tight_layout(); fig.savefig(OUT / "pump_wave.png", dpi=120); plt.show()

> 고장이 가까워지면 뾰족한 충격이 주기적으로 나타남. 닳은 베어링 위를 구름체가 지날 때마다 생기는 충격임.

## 2. 데이터 처리: 특징 추출

파형 1,024개 점을 숫자 몇 개로 줄임.

| 특징 | 뜻 | 잘 잡는 것 |
|---|---|---|
| 실효값 (RMS) | 진동의 평균적인 크기 | 전체적으로 커진 진동 (후기) |
| 첨도 | 뾰족한 충격이 섞인 정도 | 초기 베어링 결함 |
| 파고율 | 최댓값 ÷ 실효값 | 충격 |

In [ ]:
feat = pd.DataFrame(pm.vib_features(vib))
df = pd.concat([win, feat], axis=1)
fig, axes = plt.subplots(3, 1, figsize=(13, 7.5), sharex=True)
for ax, col, name in zip(axes, ["rms", "kurtosis", "current_rms_a"], ["진동 실효값 (g)", "진동 첨도", "모터 전류 (A)"]):
    for unit, g in df.groupby("unit_id"):
        ax.plot(g.timestamp, g[col].rolling(3, min_periods=1).median(), lw=1.2, label=unit)
    for _, r in fail.iterrows():
        ax.axvline(r.failure_time, color="crimson", ls="--", lw=1)
    ax.set_ylabel(name); ax.grid(alpha=0.3)
axes[0].legend(ncol=4, loc="upper left"); axes[0].set_title("특징의 추이 (빨간 점선: 고장)")
plt.tight_layout(); fig.savefig(OUT / "pump_trend.png", dpi=120); plt.show()

> 첨도가 가장 먼저 오르고, 실효값이 그 뒤를 따르고, 전류는 고장 직전에야 조금 오름. **무엇을 재느냐에 따라 알 수 있는 시점이 다름.**

## 3. 상태 검출: 정상 기준과 이상 점수

설비마다 처음 5일을 정상으로 보고 기준(평균 · 표준편차)을 잡음. 이상 점수 = 기준에서 벗어난 정도.

In [ ]:
BASE_DAYS = 5                               # 심화: 기준을 잡는 기간
KEYS = ["rms", "kurtosis", "crest"]         # 심화: 점수에 쓰는 특징
df["score"] = 0.0
for unit, g in df.groupby("unit_id"):
    base = g[g.timestamp < g.timestamp.min() + pd.Timedelta(days=BASE_DAYS)]
    hs = pm.HealthScore(KEYS).fit({k: base[k].values for k in KEYS})
    df.loc[g.index, "score"] = hs.score({k: g[k].values for k in KEYS})

fig, ax = plt.subplots(figsize=(13, 3.6))
for unit, g in df.groupby("unit_id"):
    ax.plot(g.timestamp, g.score, lw=1, label=unit)
for _, r in fail.iterrows():
    ax.axvline(r.failure_time, color="crimson", ls="--", lw=1)
ax.set_ylim(0, 40); ax.set_ylabel("이상 점수"); ax.legend(ncol=4); ax.grid(alpha=0.3)
ax.set_title("이상 점수의 추이"); plt.tight_layout(); fig.savefig(OUT / "pump_score.png", dpi=120); plt.show()

> 고장 없이 잘 돌던 구간에도 점수가 튀는 점이 있음. 지게차 충돌 같은 일시적인 충격임. 이것이 **오경보**의 재료가 됨.
> 같은 열화라도 설비마다 점수의 크기가 다름. 기준을 잡은 5일 동안 충격이 섞였던 설비는 기준이 느슨해져 점수가 낮게 나옴.

## 4. 건전성 평가: 전조를 가려내는가

고장 기록으로 구간마다 "고장까지 남은 시간"을 계산하고, 전조(고장 전 120시간 이내)와 정상(고장 300시간 이전, 고장 없는 설비)을 나눔.

In [ ]:
from sklearn.metrics import roc_auc_score

ft = fail.set_index("unit_id")["failure_time"]
df["to_fail_h"] = [(ft[u] - t).total_seconds() / 3600 if (u in ft.index and t < ft[u]) else np.nan
                   for u, t in zip(df.unit_id, df.timestamp)]
PRECURSOR_H = 120
is_pre = df.to_fail_h <= PRECURSOR_H
is_normal = df.to_fail_h.isna() | (df.to_fail_h > 300)
ev = df[is_pre | is_normal]
auroc = roc_auc_score(is_pre[ev.index], ev.score)
print(f"전조 구간 {is_pre.sum()}개,  정상 구간 {is_normal.sum()}개")
print(f"이상탐지 AUROC {auroc:.3f}   (목표 0.90 이상)")

## 5. 예지 평가: 알람 기준과 선행 시간

- **경고**: 점검을 계획함 (부품 준비, 정비 일정)
- **정지**: 설비를 세움
- 오경보 억제: 기준을 **연속으로 몇 번** 넘어야 알람을 낼지 정함

In [ ]:
WARN, STOP = 5.0, 15.0          # 심화: 경고 · 정지 기준 점수
N_CONSEC = 3                    # 심화: 연속 몇 번 넘으면 알람 (1 이면 한 번만 넘어도 알람)

def alarms(score, th, n):
    """기준을 n 번 연속으로 넘은 지점이 True"""
    over = (score >= th).astype(int)
    return over.rolling(n).sum() >= n

rows, false_warn = [], 0
for unit, g in df.groupby("unit_id"):
    warn, stop = alarms(g.score, WARN, N_CONSEC), alarms(g.score, STOP, N_CONSEC)
    false_warn += int((warn & is_normal[g.index]).sum())
    if unit in ft.index:
        before = g.timestamp < ft[unit]
        w, s = g[warn & before & (g.to_fail_h < 250)], g[stop & before & (g.to_fail_h < 250)]
        rows.append({"설비": unit, "경고 (고장 몇 시간 전)": round(w.to_fail_h.max(), 0) if len(w) else None,
                     "정지 (고장 몇 시간 전)": round(s.to_fail_h.max(), 0) if len(s) else None})
lead = pd.DataFrame(rows).set_index("설비")
recall = (alarms(df.score, WARN, 1) & is_pre).sum() / is_pre.sum()
print(f"전조 재현율 {recall:.2f}   (전조 구간 중 경고 기준을 넘은 비율, 목표 0.80 이상)")
print(f"정상 구간에서의 오경보 {false_warn}회   (연속 {N_CONSEC}회 기준)")
lead

In [ ]:
# 연속 횟수에 따른 오경보
cmp = []
for n in (1, 2, 3, 5):
    fw = sum(int((alarms(g.score, WARN, n) & is_normal[g.index]).sum()) for _, g in df.groupby("unit_id"))
    first = [g[alarms(g.score, WARN, n) & (g.timestamp < ft[u]) & (g.to_fail_h < 250)].to_fail_h.max()
             for u, g in df.groupby("unit_id") if u in ft.index]
    cmp.append({"연속 횟수": n, "정상 구간 오경보(회)": fw, "평균 경고 선행 시간(h)": round(float(np.nanmean(first)), 0)})
pd.DataFrame(cmp).set_index("연속 횟수")

> 한 번만 넘어도 알람을 내면 충격 때마다 오경보가 남. 연속 횟수를 늘리면 오경보는 사라지지만 알람이 그만큼 늦어짐.

## 6. 조치 권고: 알람 기준표

In [ ]:
rules = pd.DataFrame([
    {"단계": "정상", "기준": f"이상 점수 {WARN} 미만", "조치": "정상 운전, 2시간마다 측정"},
    {"단계": "경고", "기준": f"이상 점수 {WARN} 이상이 연속 {N_CONSEC}회", "조치": "측정 주기 단축, 베어링 준비, 다음 비가동 시간에 점검",
     "평균 선행 시간(h)": round(float(lead.iloc[:, 0].mean()), 0)},
    {"단계": "정지", "기준": f"이상 점수 {STOP} 이상이 연속 {N_CONSEC}회", "조치": "설비 정지, 베어링 교체",
     "평균 선행 시간(h)": round(float(lead.iloc[:, 1].mean()), 0)},
]).set_index("단계")
rules.to_csv(OUT / "alarm_rules.csv", encoding="utf-8-sig")
rules

### 워크북에 옮겨 적기

1. AUROC 와 전조 재현율
2. 알람 기준표 (단계 · 기준 · 조치 · 평균 선행 시간)
3. 오경보가 났을 때 현장에서 어떻게 처리할지 한 문장

### 심화

- `KEYS` 에서 첨도를 빼고 실효값만 쓰면 경고 선행 시간이 얼마나 줄어드는지
- `WARN` 을 3 으로 낮추면 선행 시간과 오경보가 어떻게 달라지는지
- 전류(`current_rms_a`)만으로 점수를 만들면 몇 시간 전에 알 수 있는지

저장된 파일: `data/session3/work/alarm_rules.csv`